# Grokking em Redes Neurais: Paridade Esparsa com Transformer

Este notebook demonstra o fenômeno de **grokking** na tarefa de **Paridade Esparsa** utilizando um modelo de **Transformer**. Esta arquitetura de atenção bidirecional aprende a filtrar os bits de ruído e focar na subestrutura lógica (XOR) que define o rótulo.

### O Problema
A entrada consiste em sequências de tokens binários de tamanho $N = 10$ com valores em $\{0, 1\}$. O rótulo $y \in \{0, 1\}$ é a paridade (multiplicação ou XOR) de apenas $k = 3$ bits específicos (os 3 primeiros bits):

$$y = x_0 \oplus x_1 \oplus x_2$$

Os 7 bits restantes agem como ruído. A rede passa por um platô metaestável (fase desordenada / memorização) onde a acurácia de validação permanece estagnada em torno de $60\%$, até que repentinamente a regularização harmônica por decaimento de pesos (*weight decay*) induz uma transição de fase de primeira ordem para generalização perfeita de $100\%$.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Configurar sementes para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando o dispositivo: {device}")

### 1. Definição do Modelo Transformer

Utilizamos um Transformer Encoder de 1 camada com atenção bidirecional e pooling médio sobre o comprimento da sequência antes de realizar a classificação final.

In [ ]:
class ParityTransformerBidirectional(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim, seq_len):
        super().__init__()
        self.embed = nn.Embedding(2, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(seq_len, embed_dim))
        
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, 2, bias=True)
        
    def forward(self, x):
        # x shape: (B, seq_len)
        B, S = x.shape
        h = self.embed(x) + self.pos_embed[:S].unsqueeze(0)
        
        # Self-Atenção
        h_norm = self.ln1(h)
        q = self.q_proj(h_norm)
        k = self.k_proj(h_norm)
        v = self.v_proj(h_norm)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, v)
        attn_out = self.out_proj(attn_out)
        
        h = h + attn_out
        
        # MLP residual
        h_norm = self.ln2(h)
        mlp_out = self.fc2(self.act(self.fc1(h_norm)))
        h = h + mlp_out
        
        # Pooling médio no tempo
        h_mean = h.mean(dim=1)
        logits = self.unembed(h_mean)
        return logits

### 2. Geração de Dados e Divisão de Treino/Validação

Mapeamos a entrada binária $\{-1, 1\}$ para tokens $\{0, 1\}$. Dividimos o dataset em apenas $70$ amostras de treino (cerca de $6.8\%$) e $954$ amostras de validação ($93.2\%$).

In [ ]:
def generate_all_data(N, k_indices):
    grids = np.meshgrid(*[[ -1.0, 1.0 ] for _ in range(N)])
    x_raw = torch.tensor(np.stack(grids, axis=-1).reshape(-1, N), dtype=torch.float32)
    y_raw = x_raw[:, k_indices].prod(dim=1)
    y = ((y_raw + 1) / 2).long() # 0 ou 1
    
    # Mapear x_raw (-1, 1) para tokens (0, 1)
    x_tokens = ((x_raw + 1) / 2).long()
    return x_tokens, y

N = 10
k_indices = [0, 1, 2]
all_x, all_y = generate_all_data(N, k_indices)

# Divisão de treino (70 amostras) e validação (954 amostras)
indices = np.random.permutation(len(all_x))
train_split = 70
train_idx = indices[:train_split]
val_idx = indices[train_split:]

x_train, y_train = all_x[train_idx], all_y[train_idx]
x_val, y_val = all_x[val_idx], all_y[val_idx]

print(f"Exemplos de Treino: {len(x_train)}")
print(f"Exemplos de Validação: {len(x_val)}")

### 3. Treinamento do Transformer

Treinamos o Transformer com AdamW, decaimento de pesos $\lambda = 0.5$ e taxa de aprendizado de $10^{-3}$ por 8000 épocas.

In [ ]:
model = ParityTransformerBidirectional(embed_dim=128, num_heads=4, hidden_dim=256, seq_len=N).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.5)

epochs = 8000
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'weight_l2': []
}

x_train, y_train = x_train.to(device), y_train.to(device)
x_val, y_val = x_val.to(device), y_val.to(device)

print("Iniciando o treinamento...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        train_acc = (logits.argmax(dim=-1) == y_train).float().mean().item()
        val_logits = model(x_val)
        val_loss = criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()
        w_l2 = sum(p.pow(2).sum().item() for p in model.parameters())
        
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['weight_l2'].append(w_l2)
    
    if (epoch + 1) % 1000 == 0 or epoch == 0:
        print(f"Época {epoch+1:5d}/{epochs} | "
              f"Perda Treino: {loss.item():.4f} | Acc Treino: {train_acc*100:6.2f}% | "
              f"Acc Val: {val_acc*100:6.2f}% | Norma L2: {w_l2:.3f}")
        
        if train_acc > 0.999 and val_acc > 0.985:
            print(f"\nGrokking atingido na época {epoch+1}!")
            break

### 4. Resultados Gráficos

Plotamos as curvas de perda, acurácia e norma dos pesos $L_2$ ao longo do treinamento.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(6.5, 9.0), sharex=True)

# 1. Perda
ax1.plot(history['train_loss'], label='Treino', color='blue', alpha=0.7)
ax1.plot(history['val_loss'], label='Validação', color='red', alpha=0.7)
ax1.set_yscale('log')
ax1.set_title('Perda vs Épocas')
ax1.set_ylabel('Perda (Log)')
ax1.legend()
ax1.grid(True, which='both', ls='--')

# 2. Acurácia
ax2.plot(history['train_acc'], label='Treino', color='blue', alpha=0.7)
ax2.plot(history['val_acc'], label='Validação', color='red', alpha=0.7)
ax2.set_title('Acurácia vs Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, ls='--')

# 3. Norma L2
ax3.plot(history['weight_l2'], label='Norma L2 (w)', color='purple', alpha=0.7)
ax3.set_title('Norma dos Pesos L2 vs Épocas')
ax3.set_xlabel('Épocas')
ax3.set_ylabel('Norma L2')
ax3.legend()
ax3.grid(True, ls='--')

plt.suptitle('Grokking no Transformer (Paridade Esparsa: N=10, k=3)', fontsize=12, y=0.99)
plt.tight_layout()

import os
os.makedirs('../paper', exist_ok=True)
plt.savefig('../paper/grokking_parity_transformer.png', dpi=300)
plt.show()